In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
dataset = pd.read_csv('Age_Salary_Buy.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

In [4]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

# Compute the means and standard deviations of features w.r.t
# training set. Transform X_train
X_train = sc.fit_transform(X_train)

# Using the scaler computed above, transform X_test
X_test = sc.transform(X_test)

In [5]:
class Value:
    """ stores a single scalar value and its gradient """

    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0
        
        self._backward = lambda: None
        self._prev = set(_children)        

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers"
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,))

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        out = Value(np.tanh(self.data), (self,))

        def _backward():
            self.grad += (1-out.data**2) * out.grad
        out._backward = _backward

        return out


    def log(self):
        eps = 1e-15
        x = np.clip(self.data, eps, 1 - eps)
        out = Value(np.log(x), (self,))
        def _backward():
            # derivative uses clamped x to avoid 1/0
            self.grad += (1/x) * out.grad
        out._backward = _backward
        return out


    def sig(self):
        out = Value(1.0/(1+np.exp(-self.data)), (self,))

        def _backward():
            self.grad += out.data*(1-out.data) * out.grad
        out._backward = _backward

        return out
    
    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self): # -self
        return self * -1

    def __radd__(self, other): # other + self
        return self + other

    def __sub__(self, other): # self - other
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


In [6]:
import random

# Activation helpers that operate on a single Value
def act_identity(v): 
    return v

def act_relu(v):
    return v.relu()

def act_tanh(v):
    return v.tanh()

def act_sigmoid(v):
    return v.sig()

class Neuron:

    def __init__(self, nin, activation=act_sigmoid):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0)        
        self.activation = activation

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return self.activation(act)

    def parameters(self):
        return self.w + [self.b]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0
    
    def __repr__(self):
        return f"Linear Neuron({len(self.w)})"

class Layer:

    def __init__(self, nin, nout, activation=act_sigmoid):
        self.neurons = [Neuron(nin, activation=activation) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0
    
    def __repr__(self):
        return f"Layer"

class MLP:

    def __init__(self, nin, nouts, activations):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], activation=activations[i]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0
    
    def __repr__(self):
        return f"MLP"

In [7]:
    def log_loss(y, p):
        losses = [-pi.log()*yi - (-pi+1).log()*(1-yi) for pi, yi in zip(p,y)]
        loss = sum(losses)/len(losses)
        return loss

In [23]:
mlp = MLP(2,[4,4,1],[act_relu,act_relu,act_sigmoid])
etha = 0.05
print(len(mlp.parameters()))

37


In [26]:
batch_size=32
for k in range(100):    
    #ri = np.random.permutation(X_train.shape[0])[:batch_size]
    Xb, yb = X_train, y_train 
    #Xb, yb = X_train[ri], y_train[ri]    
    
    # Forward pass
    ypred = [mlp(x) for x in Xb]    
    loss = log_loss(yb, ypred)

    # Bacward pass
    mlp.zero_grad()
    loss.backward()

    # Update
    for p in mlp.parameters():
        p.data += -etha*p.grad

    if k%10 == 0: print(k, loss.data)

0 0.2674981421180223
10 0.26731675832454005
20 0.2671653339055735
30 0.2670266071648589
40 0.2668942805300363
50 0.26676701435358996
60 0.2666433947212399
70 0.2665248355656775
80 0.26641096760516103
90 0.26629777233411395
